In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

/home/fkunneman/.local/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db2",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [5]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'


In [32]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'travel')
assistant.prepare_patterns(pat)
assistant.prepare_instructions(instructions_travel)
assistant.setup_rag(collection,[['qa',qa_travel],['nav',nav]])

In [34]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'passport')
assistant.prepare_instructions(instructions_passport)
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa',qa_passport],['nav',nav]])

In [35]:
assistant.chat()

Hallo! Ik ga je helpen met het maken van een afspraak voor een paspoort. Je kan altijd een vraag stellen. Om te beginnen, zeg 'ik wil starten'.


You:  start


{'ids': [['20dea474-3abc-4070-a000-539092644c6a', '3cf7e009-d7af-49f6-8874-e9ae216dbdb0', '857f9de6-484d-4463-afac-b1f63ed70150', 'c5b24c69-d012-4724-a666-aa6da236c8b7', 'c754ab7f-0c9d-409e-829c-0bde171d92df', '3dc96b3e-6003-49e3-8543-e18dc7485ad2', 'cc7c4956-4fb9-4442-88fb-30abc16c23ac', 'd311e2b7-9dde-414d-8d49-e298babfc31b', '85e431e9-7364-4b2b-9692-c97cfd9206c9', '65af222a-59e9-432e-90c4-633f6bb86068']], 'embeddings': None, 'documents': [['Start', 'Start', 'Start', 'Start', 'Start', 'Start', 'Start', 'Start', 'Start', 'Start']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': 'all', 'type': 'nav', 'action': 'start'}, {'type': 'nav', 'action': 'start', 'step_context': 'all'}, {'step_context': 'all', 'action': 'start', 'type': 'nav'}, {'type': 'nav', 'step_context': 'all', 'action': 'start'}, {'action': 'start', 'type': 'nav', 'step_context': 'all'}, {'action': 'start', 'step_context': 'all', 'type': 'nav'}, {'type': 'n

You:  waar?


{'ids': [['e2fa9d4a-4005-469d-80c4-93edf3703302', 'ec37452c-a77c-449b-af34-b12e98154752', '30412441-8673-48b1-9e25-3760d7c69df8', 'a3b12b7a-760d-4729-a760-c32ed9ec4e2b', '92d76280-fe9c-42a1-84d8-4e09f9c63768', 'fd3e31e7-d645-41fe-9afa-332a45c14d65', 'e17d8395-f38f-4342-8366-ad0645b8f728', 'e4524aad-3892-4bc3-b444-1347e714fe91', '6092caa6-e7fb-4cfa-a15c-ca5a39cbd6f4', 'a904f03f-9dc0-4049-a7d0-b4ed6a15d881']], 'embeddings': None, 'documents': [['Waar?', 'Waar?', 'Waar?', 'Waar?', 'Waar?', 'Waar?', 'Waar?', 'Waar?', 'Waar?', 'Waar?']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'answer': "De blauwe tekst 'Paspoort, ID-kaart en rijbewijs' staat links onder de blauwe tekst 'verhuizing doorgeven'", 'type': 'instruct', 'step_context': 'o'}, {'type': 'instruct', 'answer': "De blauwe tekst 'afspraak maken' staat onder de zwarte tekst 'Aanvragen'", 'step_context': 'o'}, {'answer': "Het witte veld staat onder 'persoon personen'", 'step_context

...

{'ids': [['88c665ee-78f4-4b3b-9518-9c89eb7528ad', '09541331-c48a-48d2-b589-68ee16a64af4', 'd55803da-b880-4699-aabb-b6c44111be28', '99596c65-bdf6-4091-869c-e47c09f3564f', '4993b603-0801-460a-9635-92af3e9ff833', '0ab430d6-8067-40c4-abb3-1bb9681b68d2', '06b4a34e-1495-44b9-bbd5-28e34e732ece', '462f5c95-64a6-4b16-a5c7-b54ff29ca41c', 'b3e013c9-6d5e-4e4c-94cc-481b6adea1c3', 'e9dba687-4e64-4c85-8dc5-ac0707a40bd3']], 'embeddings': None, 'documents': [['Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?', 'Waar vind ik die knop?']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': '4', 'type': 'instruct', 'answer': "De knop 'tijd' staat rechts naast de knop 'datum'"}, {'type': 'instruct', 'step_context': '1', 'answer': "Het veld 'Van' staat onder 'Waar wil

You:  stop


{'ids': [['37e144ce-63e1-4a79-8b30-f8bf9a7a3c38', '1d7d4b9f-737e-4b52-a567-89796f5b68f6', 'b084a62e-94b4-4d5c-888f-3a75bfa5b4b6', 'f6db9593-a4ab-4316-bef6-e88c81464106', 'a7cfa395-4731-4c9b-a9f7-cd602d58ae46', '9099f1db-8b25-4c93-9368-f195c596fdf1', '8b488329-8e80-49f0-a4c9-8b6ac8f7919e', 'df2d0b40-d895-4566-8cfd-b83c53f65a7a', '5d128b1f-2035-4519-8190-a5861e1aa600', '8ccc3af9-e8bd-493a-a743-543a2006316d']], 'embeddings': None, 'documents': [['Stop', 'Stop', 'Stop', 'Stop', 'Stop', 'Stop', 'Stop', 'Stop', 'Stop', 'Stop']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'Done', 'step_context': 'all', 'type': 'nav'}, {'type': 'nav', 'step_context': 'all', 'action': 'Done'}, {'type': 'nav', 'action': 'Done', 'step_context': 'all'}, {'type': 'nav', 'step_context': 'all', 'action': 'Done'}, {'step_context': 'all', 'type': 'nav', 'action': 'Done'}, {'step_context': 'all', 'action': 'Done', 'type': 'nav'}, {'type': 'nav', 'action': '

You:  ja


{'ids': [['5107c4a9-503b-4d55-ab3a-13e0aa30e59b', '38bd88e3-d612-46b2-87ec-b4455d8036f0', 'aff51201-3c86-4eba-8fa4-5adef71346e0', '6e4c258f-9156-42b6-9825-70fa51c562cb', '53ac54b3-c7ea-4881-a83f-0e3980b245ef', '613fc976-9103-4713-bb0c-d0f3ec47597d', '294a3d13-079f-4b37-87a3-d3598c33150e', 'a06fa15a-87f9-4433-a582-704871a73271', '1d9f7773-05fb-403b-8955-eb8ed76cc79a', '95fed125-8f22-4586-9fcb-96fe3bf8d84d']], 'embeddings': None, 'documents': [['Ja', 'Ja', 'Ja', 'Ja', 'Ja', 'Ja', 'Ja', 'Ja', 'Ja', 'Ja']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'Confirm', 'step_context': 'all', 'type': 'nav'}, {'action': 'Confirm', 'step_context': 'all', 'type': 'nav'}, {'type': 'nav', 'action': 'Confirm', 'step_context': 'all'}, {'step_context': 'all', 'type': 'nav', 'action': 'Confirm'}, {'action': 'Confirm', 'type': 'nav', 'step_context': 'all'}, {'step_context': 'all', 'action': 'Confirm', 'type': 'nav'}, {'type': 'nav', 'action': 'Co